# El Lado Oscuro de las Pruebas de Integración

**Objetivo:** Reflexionar sobre las limitaciones de las pruebas de integración, los antipatrones que generan falsa confianza, y la diferencia entre cobertura de código y cobertura de integración.

---

## 1. La paradoja de los tests verdes

Las pruebas iniciales del proyecto pasan. El coverage es alto. Y sin embargo, el sistema tiene errores graves.

¿Cómo es posible?

Porque las pruebas son **demasiado permisivas**:
- Solo verifican que la función retorna un valor, **no que el estado sea correcto**.
- No inyectan fallos en las dependencias.
- No verifican los efectos secundarios en el almacenamiento.

Esta es la **paradoja de los tests verdes**: una suite de pruebas puede pasar al 100% y el sistema puede fallar estrepitosamente en producción.

---

## 2. Demostración: el impostor

In [ ]:
import sys, os, tempfile, json
sys.path.insert(0, os.path.abspath('..'))

from src.storage import TaskStorage
from src.service import TaskService

class StubNotifier:
    def __init__(self, fail=False):
        self._fail = fail
    def send(self, message):
        if self._fail: raise ConnectionError("Fallo simulado")

# ─────────────────────────────────────────────────────
# VERSIÓN SABOTEADA de TaskService
# add_task() siempre retorna True sin guardar ni notificar.
# Reproduce el ejercicio de sabotaje del README.
# ─────────────────────────────────────────────────────
class ImpostorService:
    def __init__(self, storage, notifier):
        self.storage = storage
        self.notifier = notifier

    def add_task(self, title):
        return True  # IMPOSTOR: no guarda, no notifica

    def complete_task(self, title):
        tasks = self.storage.load()
        for t in tasks:
            if t['title'] == title:
                t['done'] = True
                self.storage.save(tasks)
                return True
        return False

    def list_tasks(self):
        return self.storage.load()


def make_temp_storage():
    f = tempfile.NamedTemporaryFile(suffix='.json', delete=False)
    f.close()
    os.unlink(f.name)
    return TaskStorage(f.name), f.name


# Prueba DÉBIL (igual que las del repositorio inicial)
def test_debil(service_class):
    storage, path = make_temp_storage()
    try:
        service = service_class(storage, StubNotifier())
        result = service.add_task("Comprar leche")
        assert result is True  # ← aserción débil
        return "PASA"
    except AssertionError:
        return "FALLA"
    finally:
        if os.path.exists(path): os.unlink(path)


print("Prueba DÉBIL con servicio REAL:     ", test_debil(TaskService))
print("Prueba DÉBIL con servicio IMPOSTOR: ", test_debil(ImpostorService))
print()
print("RESULTADO: ambas pasan. La prueba débil no detecta el impostor.")

In [ ]:
# Prueba FUERTE: verifica el estado del sistema, no solo el retorno
def test_fuerte(service_class):
    storage, path = make_temp_storage()
    try:
        service = service_class(storage, StubNotifier())
        result = service.add_task("Comprar leche")

        assert result is True                        # valor de retorno
        tasks = service.list_tasks()
        assert len(tasks) == 1                       # tarea existe
        assert tasks[0]["title"] == "Comprar leche" # título correcto
        assert tasks[0]["done"] is False             # estado inicial correcto
        with open(path) as f:                        # persiste en disco
            disk = json.load(f)
        assert disk == [{"title": "Comprar leche", "done": False}]
        return "PASA"
    except AssertionError as e:
        return f"FALLA ({e})"
    finally:
        if os.path.exists(path): os.unlink(path)


print("Prueba FUERTE con servicio REAL:     ", test_fuerte(TaskService))
print("Prueba FUERTE con servicio IMPOSTOR: ", test_fuerte(ImpostorService))
print()
print("RESULTADO: la prueba fuerte distingue al impostor del servicio real.")
print("Verificar el ESTADO del sistema es esencial en pruebas de integración.")

---

## 3. Cobertura de código vs. cobertura de integración

La **cobertura de código** mide qué porcentaje de líneas fueron ejecutadas. Es la métrica más común, pero puede ser engañosa.

```
Cobertura de código = (Líneas ejecutadas / Total de líneas) × 100
```

La **cobertura de integración** evalúa si los *puntos de interacción* entre módulos están siendo probados:
- ¿Se probó la llamada con datos válidos **y** con datos inválidos?
- ¿Se probó el caso donde la dependencia falla?
- ¿Se verificó que los datos transferidos mantienen su formato?

**Ejercicio mental:** Corre `pytest --cov=src tests/` con las pruebas iniciales. Obtendrás una cobertura alta. Pero si modificas `add_task()` para no llamar a `notifier.send()`, la cobertura *no cambiará* si hay otro test que ejecute esa línea de otra forma. Las pruebas débiles crean una ilusión de calidad.

In [ ]:
# Ejecuta la cobertura de las pruebas iniciales
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/", "--cov=../src",
     "--cov-report=term-missing", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print()
print("Observa: la cobertura puede ser alta aunque las pruebas sean débiles.")
print("La cobertura mide EJECUCIÓN, no CORRECCIÓN.")

---

## 4. Antipatrones comunes en pruebas de integración

### ❌ Aserciones débiles
Solo comprobar que el método no lanzó excepción, o solo su valor de retorno.
```python
# MAL:
def test_add_task():
    service.add_task("T")  # no verifica nada

# BIEN:
def test_add_task():
    result = service.add_task("T")
    assert result is True
    assert service.list_tasks() == [{"title": "T", "done": False}]
```

### ❌ Sin simulación de fallos
No probar qué ocurre cuando una dependencia falla.
```python
# MAL: solo escenario feliz
service = TaskService(storage, Notifier())  # notifier real con fallo aleatorio

# BIEN: también el escenario de error
service_fallo = TaskService(storage, StubNotifier(fail=True))
```

### ❌ Estado compartido entre tests
Los tests no limpian el estado (archivo JSON) y se contaminan mutuamente.
```python
# MAL: mismo archivo para todos los tests
storage = TaskStorage("test.json")  # persiste entre tests

# BIEN: archivo temporal por test
with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as f:
    storage = TaskStorage(f.name)
```

### ❌ Uso excesivo de mocks
Reemplazar todos los módulos con mocks convierte la prueba de integración en un test unitario disfrazado. Si todo está mockeado, no estás probando integración.

---

## 5. Actividad: Sabotaje controlado

Realiza el siguiente experimento y documenta los resultados en tu informe:

1. Abre `src/service.py` y modifica `add_task()` para que **no llame** a `self.notifier.send(...)`.
2. Ejecuta las pruebas iniciales. ¿Detectaron el cambio?
3. Ahora escribe una prueba que use un `StubNotifier` con registro de mensajes y verifique que `send()` fue llamado. ¿Detecta el error?
4. Restaura el código original y ejecuta tu nueva prueba. ¿Pasa?

**Conclusión:** Las pruebas iniciales eran insuficientes; necesitamos pruebas que validen las *interacciones*, no solo los *resultados*.

---

## 6. Reflexión final

Responde en tu informe:

1. **Cobertura y calidad:** ¿Qué aprendiste sobre la diferencia entre cobertura de código y calidad de las pruebas?

2. **Estado inconsistente:** El bug del `Notifier` (tarea guardada sin notificación) es un ejemplo de *estado inconsistente*. ¿Cómo lo corregirías en el código? ¿Qué patrones de diseño (p.ej. transacción, rollback) aplicarías?

3. **Combinación de enfoques:** ¿Por qué es importante combinar los distintos enfoques de integración en lugar de elegir solo uno?

4. **Microservicios:** ¿Cómo aplicarías estos conceptos (stubs, drivers, enfoques de integración) en un proyecto real con microservicios? ¿Qué complicaciones adicionales surgen al probar la integración entre servicios independientes que se comunican por HTTP o mensajería?